# CIFAR-10 ViT-30k: Raw Weights vs NF-Smoother BigVAE Latents

This notebook compares two optimization parameterizations for the same compact CIFAR-10 ViT:

- `raw`: train all ViT weights directly.
- `nf_smoother_latent`: train BigVAE latent slots in post-hoc normalizing-flow coordinates `z_prime`; weights are decoded as `decoder_vae(flow.inverse(z_prime))`.

Set `BIG_VAE_CHECKPOINT` and `NF_SMOOTHER_CHECKPOINT` before running. The ViT config below is about 30k raw trainable parameters.

In [ ]:
from __future__ import annotations

import json
import math
import os
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

import matplotlib.pyplot as plt

try:
    from torchvision import datasets, transforms
except Exception as exc:
    datasets = None
    transforms = None
    print('torchvision unavailable:', repr(exc))

from big_vae.eval.vit_tiny_latent_optimization import (
    BigVAELatentTensorStore,
    FunctionalViTTiny,
    ViTTinyConfig,
    count_trainable_parameters,
    load_frozen_big_vae_decoder,
    make_initial_tensors,
    seed_everything,
)
from post_train_research.big_vae_heldout_eval.evaluate_parts.decoder_adapter import (
    load_latent_flattening_flow_from_checkpoint,
)

try:
    torch.set_float32_matmul_precision('high')
except Exception:
    pass

@dataclass
class NotebookConfig:
    seed: int = int(os.getenv('SEED', '42'))
    device: str = os.getenv('DEVICE', 'cuda:0' if torch.cuda.is_available() else 'cpu')
    data_dir: str = os.getenv('CIFAR10_DATA_DIR', './data/cifar10')
    download: bool = os.getenv('DOWNLOAD', 'true').strip().lower() in {'1', 'true', 'yes'}
    artifact_dir: str = os.getenv('ARTIFACT_DIR', './artifacts/loss_landscape_analysis/cifar10_vit30k_nf_smoother')

    big_vae_checkpoint: str = os.getenv('BIG_VAE_CHECKPOINT', '')
    nf_smoother_checkpoint: str = os.getenv(
        'NF_SMOOTHER_CHECKPOINT',
        os.getenv('EVAL_DECODER_ADAPTER_CHECKPOINT', ''),
    )

    train_subset: int = int(os.getenv('TRAIN_SUBSET', '10000'))
    test_subset: int = int(os.getenv('TEST_SUBSET', '2000'))
    batch_size: int = int(os.getenv('BATCH_SIZE', '256'))
    eval_batch_size: int = int(os.getenv('EVAL_BATCH_SIZE', '512'))
    num_workers: int = int(os.getenv('NUM_WORKERS', '2'))
    max_steps: int = int(os.getenv('MAX_STEPS', '400'))
    eval_every: int = int(os.getenv('EVAL_EVERY', '25'))
    amp: bool = os.getenv('AMP', 'true').strip().lower() in {'1', 'true', 'yes'}

    raw_lr: float = float(os.getenv('RAW_LR', '1e-3'))
    latent_lr: float = float(os.getenv('LATENT_LR', '1e-2'))
    weight_decay: float = float(os.getenv('WEIGHT_DECAY', '0.0'))
    betas: tuple[float, float] = (0.9, 0.999)
    eps: float = 1e-9
    grad_clip_norm: float = float(os.getenv('GRAD_CLIP_NORM', '1.0'))

    big_vae_decode: str = os.getenv('BIG_VAE_DECODE', 'weights')
    big_vae_tile_T_patches: int = int(os.getenv('BIG_VAE_TILE_T_PATCHES', '4'))
    big_vae_tile_d_out: int = int(os.getenv('BIG_VAE_TILE_D_OUT', '64'))
    latent_init: str = os.getenv('LATENT_INIT', 'base')
    latent_init_fit_steps: int = int(os.getenv('LATENT_INIT_FIT_STEPS', '0'))
    latent_init_fit_lr: float = float(os.getenv('LATENT_INIT_FIT_LR', '1e-2'))

cfg = NotebookConfig()
device = torch.device(cfg.device)
artifact_dir = Path(cfg.artifact_dir).expanduser().resolve()
artifact_dir.mkdir(parents=True, exist_ok=True)

vit_cfg = ViTTinyConfig(
    image_size=32,
    patch_size=8,
    in_channels=3,
    num_classes=10,
    hidden_dim=48,
    depth=1,
    num_heads=4,
    mlp_ratio=2.0,
    dropout=0.0,
    attention_dropout=0.0,
)

seed_everything(cfg.seed)
np.random.seed(cfg.seed)
random.seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)

print('device:', device)
print('artifact_dir:', artifact_dir)
print('cfg:', cfg)
print('vit_cfg:', vit_cfg)

In [ ]:
def require_file(path: str, label: str) -> Path:
    resolved = Path(path).expanduser().resolve()
    if not resolved.is_file():
        raise FileNotFoundError(f'{label} does not exist: {resolved}')
    return resolved

def parameter_count_from_tensors(tensors: dict[str, torch.Tensor]) -> int:
    return int(sum(int(t.numel()) for t in tensors.values()))

def autocast_context():
    if device.type == 'cuda' and cfg.amp:
        return torch.autocast(device_type='cuda', dtype=torch.float16)
    return torch.autocast(device_type='cpu', enabled=False)

@torch.no_grad()
def evaluate_model(model: torch.nn.Module, loader: DataLoader, *, max_batches: int | None = None) -> dict[str, float]:
    model.eval()
    loss_sum = 0.0
    correct = 0
    examples = 0
    for batch_idx, (images, labels) in enumerate(loader):
        if max_batches is not None and batch_idx >= int(max_batches):
            break
        images = images.to(device=device, non_blocking=True)
        labels = labels.to(device=device, non_blocking=True)
        with autocast_context():
            logits = model(images)
            loss = F.cross_entropy(logits, labels, reduction='sum')
        loss_sum += float(loss.detach().cpu().item())
        correct += int((logits.argmax(dim=-1) == labels).detach().sum().cpu().item())
        examples += int(labels.numel())
    return {'loss': loss_sum / max(1, examples), 'accuracy': correct / max(1, examples), 'examples': examples}

def grad_norm(parameters) -> float:
    total = 0.0
    for param in parameters:
        if param.grad is None:
            continue
        total += float(param.grad.detach().float().pow(2).sum().cpu().item())
    return float(math.sqrt(total))

def trainable_parameters(model: torch.nn.Module) -> list[torch.nn.Parameter]:
    return [p for p in model.parameters() if p.requires_grad]

def build_optimizer(model: torch.nn.Module, *, lr: float) -> torch.optim.Optimizer:
    return torch.optim.AdamW(
        trainable_parameters(model),
        lr=float(lr),
        weight_decay=float(cfg.weight_decay),
        betas=tuple(float(v) for v in cfg.betas),
        eps=float(cfg.eps),
    )

def train_one_model(model: torch.nn.Module, optimizer: torch.optim.Optimizer, *, label: str) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    params = trainable_parameters(model)
    start_time = time.time()
    schedule_pos = 0
    for step in range(1, int(cfg.max_steps) + 1):
        model.train()
        batch_indices = train_schedule[schedule_pos % len(train_schedule)]
        schedule_pos += 1
        images, labels = fetch_batch(train_dataset, batch_indices)
        optimizer.zero_grad(set_to_none=True)
        with autocast_context():
            logits = model(images)
            loss = F.cross_entropy(logits, labels)
        loss.backward()
        gnorm = grad_norm(params)
        if float(cfg.grad_clip_norm) > 0.0:
            torch.nn.utils.clip_grad_norm_(params, float(cfg.grad_clip_norm))
        optimizer.step()

        if step == 1 or step % int(cfg.eval_every) == 0 or step == int(cfg.max_steps):
            train_eval = evaluate_model(model, train_eval_loader)
            test_eval = evaluate_model(model, test_loader)
            row = {
                'label': label,
                'step': int(step),
                'train_loss_batch': float(loss.detach().cpu().item()),
                'train_eval_loss': float(train_eval['loss']),
                'train_eval_accuracy': float(train_eval['accuracy']),
                'test_loss': float(test_eval['loss']),
                'test_accuracy': float(test_eval['accuracy']),
                'grad_norm': float(gnorm),
                'elapsed_s': float(time.time() - start_time),
            }
            rows.append(row)
            print(
                f"{label} step={step:05d} train_acc={row['train_eval_accuracy']:.4f} "
                f"test_acc={row['test_accuracy']:.4f} test_loss={row['test_loss']:.4f} grad={gnorm:.3g}",
                flush=True,
            )
    return pd.DataFrame(rows)

def fit_latent_init_to_initial_weights(model: torch.nn.Module, target_tensors: dict[str, torch.Tensor]) -> list[dict[str, float]]:
    target = getattr(model, '_orig_mod', model)
    store = getattr(target, 'store', None)
    if not isinstance(store, BigVAELatentTensorStore) or int(cfg.latent_init_fit_steps) <= 0:
        return []
    opt = torch.optim.AdamW(store.latent_slots.parameters(), lr=float(cfg.latent_init_fit_lr), weight_decay=0.0)
    targets = {
        name: store.target_matrix(name, target_tensors[name]).to(next(store.latent_slots.parameters()).device)
        for name in store.decoded_tensor_names()
    }
    denom = max(1, sum(int(v.numel()) for v in targets.values()))
    rows = []
    for step in range(1, int(cfg.latent_init_fit_steps) + 1):
        opt.zero_grad(set_to_none=True)
        decoded = store.decode_all_matrices()
        loss = sum(F.mse_loss(decoded[name], target, reduction='sum') for name, target in targets.items()) / denom
        loss.backward()
        opt.step()
        if step == 1 or step % 25 == 0 or step == int(cfg.latent_init_fit_steps):
            value = float(loss.detach().cpu().item())
            rows.append({'step': step, 'decoded_weight_mse': value})
            print(f'latent init fit step={step} decoded_weight_mse={value:.6e}', flush=True)
    return rows

In [ ]:
if datasets is None or transforms is None:
    raise RuntimeError('torchvision is required for CIFAR-10')

CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

train_full = datasets.CIFAR10(root=cfg.data_dir, train=True, download=cfg.download, transform=transform)
test_full = datasets.CIFAR10(root=cfg.data_dir, train=False, download=cfg.download, transform=transform)
train_dataset = Subset(train_full, list(range(min(cfg.train_subset, len(train_full))))) if cfg.train_subset > 0 else train_full
test_dataset = Subset(test_full, list(range(min(cfg.test_subset, len(test_full))))) if cfg.test_subset > 0 else test_full

train_eval_loader = DataLoader(
    train_dataset,
    batch_size=int(cfg.eval_batch_size),
    shuffle=False,
    num_workers=int(cfg.num_workers),
    pin_memory=(device.type == 'cuda'),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=int(cfg.eval_batch_size),
    shuffle=False,
    num_workers=int(cfg.num_workers),
    pin_memory=(device.type == 'cuda'),
)

def build_batch_schedule(dataset_len: int, batch_size: int, steps: int, seed: int) -> list[torch.Tensor]:
    generator = torch.Generator(device='cpu')
    generator.manual_seed(int(seed))
    schedule: list[torch.Tensor] = []
    while len(schedule) < int(steps):
        perm = torch.randperm(int(dataset_len), generator=generator)
        for start in range(0, int(dataset_len), int(batch_size)):
            schedule.append(perm[start:start + int(batch_size)].clone())
            if len(schedule) >= int(steps):
                break
    return schedule

def fetch_batch(dataset, indices: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    images = []
    labels = []
    for idx in indices.tolist():
        image, label = dataset[int(idx)]
        images.append(image)
        labels.append(int(label))
    return (
        torch.stack(images, dim=0).to(device=device, non_blocking=False),
        torch.tensor(labels, dtype=torch.long, device=device),
    )

train_schedule = build_batch_schedule(len(train_dataset), cfg.batch_size, cfg.max_steps, cfg.seed)
print('train examples:', len(train_dataset), 'test examples:', len(test_dataset), 'scheduled steps:', len(train_schedule))

In [ ]:
big_vae_checkpoint = require_file(cfg.big_vae_checkpoint, 'BIG_VAE_CHECKPOINT')
nf_smoother_checkpoint = require_file(cfg.nf_smoother_checkpoint, 'NF_SMOOTHER_CHECKPOINT')

initial_tensors = make_initial_tensors(vit_cfg, seed=cfg.seed)
raw_param_count = parameter_count_from_tensors(initial_tensors)
print('raw tensor parameter count:', raw_param_count)

raw_model = FunctionalViTTiny(
    vit_cfg,
    initial_tensors,
    parameter_mode='direct',
).to(device)

big_vae = load_frozen_big_vae_decoder(str(big_vae_checkpoint), device=device)
decoder_flow, flow_cfg, flow_payload = load_latent_flattening_flow_from_checkpoint(
    checkpoint_path=nf_smoother_checkpoint,
    model=big_vae,
    device=device,
)

latent_model = FunctionalViTTiny(
    vit_cfg,
    initial_tensors,
    parameter_mode='bigvae_latent',
    big_vae=big_vae,
    big_vae_decoder_flow=decoder_flow,
    big_vae_latent_init=str(cfg.latent_init),
    big_vae_latent_space='decoder_z',
    big_vae_latent_parameterization='euclidean',
    big_vae_decode=str(cfg.big_vae_decode),
    big_vae_tile_T_patches=int(cfg.big_vae_tile_T_patches),
    big_vae_tile_d_out=int(cfg.big_vae_tile_d_out),
).to(device)
latent_init_fit_rows = fit_latent_init_to_initial_weights(latent_model, initial_tensors)

store = latent_model.store
print('raw trainable params:', count_trainable_parameters(raw_model))
print('latent trainable params:', count_trainable_parameters(latent_model))
print('latent slots params:', store.latent_numel())
print('bigvae decoded params:', store.big_vae_decoded_numel())
print('decoded tile count:', store.decoded_tile_count(), 'tile shape:', store.tile_decode_shape())
print('flow config:', flow_cfg)

with torch.no_grad():
    base_latent = big_vae.latent_base.detach().reshape(1, -1).to(device=device, dtype=torch.float32)
    flow_base = decoder_flow(base_latent)[0]
    first_slot = next(iter(store.latent_slots.values())).detach().reshape(1, -1).to(dtype=torch.float32)
    print('first latent slot equals flow(base_z) mse:', float((first_slot - flow_base).pow(2).mean().detach().cpu().item()))

In [ ]:
raw_optimizer = build_optimizer(raw_model, lr=cfg.raw_lr)
latent_optimizer = build_optimizer(latent_model, lr=cfg.latent_lr)

print('initial raw eval:', evaluate_model(raw_model, test_loader))
print('initial latent eval:', evaluate_model(latent_model, test_loader))

raw_history = train_one_model(raw_model, raw_optimizer, label='raw')
latent_history = train_one_model(latent_model, latent_optimizer, label='nf_smoother_latent')

history = pd.concat([raw_history, latent_history], ignore_index=True)
history_path = artifact_dir / 'raw_vs_nf_smoother_latent_history.csv'
history.to_csv(history_path, index=False)
print('saved history:', history_path)
display(history.tail(10))

In [ ]:
summary = {
    'config': asdict(cfg),
    'vit_cfg': asdict(vit_cfg),
    'raw_param_count': int(raw_param_count),
    'raw_trainable_params': int(count_trainable_parameters(raw_model)),
    'latent_trainable_params': int(count_trainable_parameters(latent_model)),
    'latent_slot_params': int(store.latent_numel()),
    'bigvae_decoded_params': int(store.big_vae_decoded_numel()),
    'decoded_tile_count': int(store.decoded_tile_count()),
    'decoded_tile_shape': tuple(int(v) for v in store.tile_decode_shape()),
    'big_vae_checkpoint': str(big_vae_checkpoint),
    'nf_smoother_checkpoint': str(nf_smoother_checkpoint),
    'flow': {
        'dim': int(flow_cfg.dim),
        'num_layers': int(flow_cfg.num_layers),
        'hidden_dim': int(flow_cfg.hidden_dim),
        'network_depth': int(flow_cfg.network_depth),
        'log_scale_clamp': float(flow_cfg.log_scale_clamp),
        'dropout': float(flow_cfg.dropout),
    },
    'final': history.sort_values(['label', 'step']).groupby('label').tail(1).to_dict(orient='records'),
    'latent_init_fit': latent_init_fit_rows,
}
summary_path = artifact_dir / 'raw_vs_nf_smoother_latent_summary.json'
summary_path.write_text(json.dumps(summary, indent=2, sort_keys=True), encoding='utf-8')
print('saved summary:', summary_path)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for label, frame in history.groupby('label'):
    axes[0].plot(frame['step'], frame['test_accuracy'], marker='o', label=label)
    axes[1].plot(frame['step'], frame['test_loss'], marker='o', label=label)
axes[0].set_title('CIFAR-10 test accuracy')
axes[0].set_xlabel('step')
axes[0].set_ylabel('accuracy')
axes[0].grid(True, alpha=0.25)
axes[1].set_title('CIFAR-10 test loss')
axes[1].set_xlabel('step')
axes[1].set_ylabel('cross entropy')
axes[1].grid(True, alpha=0.25)
for ax in axes:
    ax.legend()
plot_path = artifact_dir / 'raw_vs_nf_smoother_latent_curves.png'
fig.savefig(plot_path, dpi=160)
print('saved plot:', plot_path)
plt.show()